<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [6]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [7]:
response = requests.get(spacex_url)

Check the content of the response


In [8]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 521: Web server is down</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-black

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [9]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [10]:
response=requests.get(static_json_url)

In [11]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [12]:
# Use json_normalize meethod to convert the json result into a dataframe


Using the dataframe <code>data</code> print the first 5 rows


In [13]:
# Decode the response content as JSON
data = response.json()

# Convert the JSON data into a Pandas DataFrame
df = pd.json_normalize(data)

# Display the first few rows to verify
df.head()


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [15]:
import pandas as pd
import datetime

# 1. Asegúrate de que 'data' sea un DataFrame
# Si 'data' es el resultado de response.json(), usa esto:
df = pd.json_normalize(data)

# 2. Ahora usa 'df' para seleccionar las columnas
data = df[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# 3. Filtramos las filas (usamos .copy() para evitar advertencias de SettingWithCopy)
data = data[data['cores'].map(len) == 1].copy()
data = data[data['payloads'].map(len) == 1].copy()

# 4. Extraemos el primer valor de las listas
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# 5. Convertimos la fecha
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# 6. Filtramos por fecha
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [16]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [17]:
# 1. Inicializa la lista vacía
BoosterVersion = []

# 2. Define la función (ejemplo genérico de cómo se suele estructurar en este laboratorio)
def getBoosterVersion(data):
    for x in data['rocket']:
        # Aquí suele haber una llamada a la API de SpaceX para obtener el nombre del cohete
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x))
        if response.status_code == 200:
            version = response.json()['name']
            BoosterVersion.append(version)
        else:
            BoosterVersion.append(None)

# 3. Llama a la función pasando tu DataFrame 'data'
getBoosterVersion(data)

# 4. Ahora 'BoosterVersion' debería tener datos
print(BoosterVersion[:5]) # Muestra los primeros 5 elementos

[None, None, None, None, None]


Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [18]:
# Llamada a la función para llenar la lista BoosterVersion
getBoosterVersion(data)

# Verificación: imprime los primeros 5 elementos para asegurar que funcionó
print(BoosterVersion[0:5])

[None, None, None, None, None]


the list has now been update 


In [20]:
# Reiniciar la lista para que comience vacía
BoosterVersion = []

# Volver a ejecutar la función asegurando que uses el DataFrame 'data' actual
getBoosterVersion(data)

# Verificar las longitudes antes de asignar
print(f"Longitud de la lista: {len(BoosterVersion)}")
print(f"Número de filas en el DataFrame: {len(data)}")

# Solo si ambos números coinciden, ejecuta la asignación:
if len(BoosterVersion) == len(data):
    data['BoosterVersion'] = BoosterVersion
    print("¡Asignación exitosa!")
else:
    print("Error: Aún no coinciden. Revisa si 'data' ha cambiado.")

Longitud de la lista: 94
Número de filas en el DataFrame: 94
¡Asignación exitosa!


we can apply the rest of the  functions here:


In [22]:
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            # 1. Hacemos la petición
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x))
            
            # 2. Verificamos que la petición fue exitosa (código 200)
            if response.status_code == 200:
                # 3. Decodificamos solo si es exitoso
                res_json = response.json()
                Longitude.append(res_json['longitude'])
                Latitude.append(res_json['latitude'])
            else:
                # En caso de error, agregamos None o valores por defecto
                Longitude.append(None)
                Latitude.append(None)

In [24]:
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            # 1. Realizar la petición
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+str(load))
            
            # 2. Verificar que la respuesta sea exitosa antes de procesar
            if response.status_code == 200:
                res_json = response.json()
                PayloadMass.append(res_json.get('mass_kg'))
                Orbit.append(res_json.get('orbit'))
            else:
                # 3. Manejar casos donde la petición falle
                PayloadMass.append(None)
                Orbit.append(None)

In [26]:
def getCoreData(data):
    for core in data['cores']:
        if core is not None:
            # 1. Realizar la petición
            url = "https://api.spacexdata.com/v4/cores/" + str(core)
            response = requests.get(url)
            
            # 2. Verificar que la respuesta fue exitosa (200)
            if response.status_code == 200:
                res_json = response.json()
                Block.append(res_json.get('block'))
                ReusedCount.append(res_json.get('reuse_count'))
            else:
                # 3. Manejar errores agregando None para mantener el tamaño de las listas
                Block.append(None)
                ReusedCount.append(None)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [27]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [30]:
# Ejemplo: si la longitud máxima es 100 y una lista tiene 90
target_length = 100 
for key in launch_dict:
    while len(launch_dict[key]) < target_length:
        launch_dict[key].append(None)


Show the summary of the dataframe


In [31]:
# 1. Get a concise summary of the DataFrame structure
df.info()

# 2. Get statistical summary of numerical columns
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   static_fire_date_utc       100 non-null    object 
 1   static_fire_date_unix      100 non-null    float64
 2   tbd                        107 non-null    bool   
 3   net                        107 non-null    bool   
 4   window                     100 non-null    float64
 5   rocket                     107 non-null    object 
 6   success                    107 non-null    bool   
 7   details                    100 non-null    object 
 8   crew                       107 non-null    object 
 9   ships                      107 non-null    object 
 10  capsules                   107 non-null    object 
 11  payloads                   107 non-null    object 
 12  launchpad                  107 non-null    object 
 13  auto_update                107 non-null    bool   

,static_fire_date_unix,window,flight_number,date_unix,fairings
count,1.000000e+02,100.000000,107.000000,1.070000e+02,0.0
mean,1.499305e+09,2553.900000,54.000000,1.494051e+09,NaN
std,8.414133e+07,4108.909732,31.032241,9.692164e+07,NaN
min,1.142554e+09,0.000000,1.000000,1.143239e+09,NaN
25%,1.465406e+09,0.000000,27.500000,1.458641e+09,NaN
50%,1.516898e+09,0.000000,54.000000,1.517434e+09,NaN
75%,1.556707e+09,5595.000000,80.500000,1.560891e+09,NaN
max,1.605111e+09,18300.000000,107.000000,1.605486e+09,NaN


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [34]:
# 1. Asegúrate de definir la lista vacía JUSTO antes de llamar a la función
BoosterVersion = [] 

# 2. Ejecuta la función de llenado (asegúrate de pasar el DataFrame correcto, ej: 'data')
getBoosterVersion(data)

# 3. VERIFICACIÓN CRÍTICA:
print(f"Longitud de la lista: {len(BoosterVersion)}")
print(f"Longitud del DataFrame: {len(data)}")

# 4. Asignación segura (solo si las longitudes coinciden exactamente)
if len(BoosterVersion) == len(data):
    data['BoosterVersion'] = BoosterVersion
    print("¡Asignación exitosa!")
    # Ahora puedes filtrar
    data_falcon9 = data[data['BoosterVersion'] == 'Falcon 9'].copy()
    print("Filtro aplicado correctamente.")
else:
    print("¡Error! Las longitudes aún no coinciden. Revisa el paso 2.")

Longitud de la lista: 94
Longitud del DataFrame: 94
¡Asignación exitosa!
Filtro aplicado correctamente.


Now that we have removed some values we should reset the FlgihtNumber column


In [35]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,rocket,payloads,launchpad,cores,flight_number,date_utc,date,BoosterVersion,FlightNumber


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [36]:
data_falcon9.isnull().sum()

rocket            0.0
payloads          0.0
launchpad         0.0
cores             0.0
flight_number     0.0
date_utc          0.0
date              0.0
BoosterVersion    0.0
FlightNumber      0.0
dtype: float64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [44]:
target_len = len(data)

# Función rápida para ajustar cualquier lista que sea más corta
def ajustar_lista(lista, target):
    while len(lista) < target:
        lista.append(None)
    return lista

# Aplica esto a todas tus listas antes de crear el DataFrame:
BoosterVersion = ajustar_lista(BoosterVersion, target_len)
PayloadMass = ajustar_lista(PayloadMass, target_len)
# ... repite para todas

!pip install openpyxl

# Asegúrate de usar el DataFrame que ya contiene todos los datos alineados (df)
# El parámetro index=False es importante para no incluir la columna de números de fila de Pandas
df.to_excel('dataset_spacex.xlsx', index=False, engine='openpyxl')

print("Archivo 'dataset_spacex.xlsx' creado exitosamente.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 10.5 MB/s eta 0:00:00
Archivo 'dataset_spacex.xlsx' creado exitosamente.


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
